In [1]:
import polars as pl
import pickle
import os, sys
from transformers import AutoTokenizer, AutoModel  
#from owlready2 import *

sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from nlp_embeddings import eval_embeddings

import numpy as np


In [2]:
disease_ot = pl.read_parquet("../data/disease/disease.parquet")
disease_ot.head(1)

id,code,name,description,dbXRefs,parents,exactSynonyms,relatedSynonyms,narrowSynonyms,broadSynonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology,synonyms
str,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],struct[3],struct[4]
"""DOID_0050890""","""http://purl.obolibrary.org/obo…","""synucleinopathy""","""A neurodegenerative disease th…","[""MEDGEN:1682194"", ""MESH:D000080874"", … ""UMLS:C5191670""]","[""EFO_0005772"", ""MONDO_0021179""]","[""alpha Synucleinopathies"", ""synucleinopathy""]","[""alpha synucleinopathies"", ""synucleinopathies""]",[],[],[],[],"[""EFO_0006792"", ""EFO_1001050""]","[""EFO_0005772"", ""MONDO_0021179"", … ""OTAR_0000020""]","[""EFO_0000618"", ""OTAR_0000020""]","[""EFO_0006792"", ""EFO_1001050"", … ""MONDO_0014835""]","{false,false,{""http://purl.obolibrary.org/obo/DOID_0050890"",""DOID_0050890""}}","{[""alpha Synucleinopathies"", ""synucleinopathy""],[""alpha synucleinopathies"", ""synucleinopathies""],[],[]}"


In [3]:
def read_ot_parquet(folder):
    files = [f for f in os.listdir(folder) if '.parquet' in f]
    dfs = [pl.read_parquet(folder + "/" + f) for f in files]
    # Concatenate the DataFrames vertically (axis=0)
    polar_df = pl.concat(dfs)
    return polar_df

In [4]:
# dbXRefs are always different from the diseaseId
(
    disease_ot
    .explode(pl.col("dbXRefs"))
    .select(["id","dbXRefs"])
    .with_columns((pl.col("id") == pl.col("dbXRefs")).alias("equal"))
)['equal'].any()

False

In [5]:
# name and synonyms
(
    disease_ot
    .explode(pl.col("exactSynonyms"))
    .select(["name","exactSynonyms"])
    .with_columns((pl.col("name") == pl.col("exactSynonyms")).alias("equal"))
)['equal'].any()

# True --> some synonyms are equal to the name

True

In [6]:
ot_associations = read_ot_parquet("../data/association_by_datasource_direct")
ot_associations.head(1)

diseaseId,targetId,aggregationType,aggregationValue,associationScore,evidenceCount,timeseries,currentNovelty
str,str,str,str,f64,i64,list[struct[4]],f64
"""DOID_0050890""","""ENSG00000070610""","""datasourceId""","""europepmc""",0.034808,3,"[{2024,0.034808,0.001521,null}, {2014,0.0,0.0,null}, … {2021,0.033456,0.003034,1}]",0.000161


In [7]:
ot_associations.shape

(4732211, 8)

In [8]:
ot_associations_diseases = ot_associations['diseaseId'].unique().to_list()
print(len(ot_associations_diseases))

26288


Data structure for vector embeddings storage v0.1
```
{
    Label   -> embedding
            -> Id
}
```

In [9]:
disease_names_df = (
    disease_ot
    .explode(pl.col("exactSynonyms"))
    .select(['id','name','exactSynonyms'])
    .unpivot(['name','exactSynonyms'], index="id")
    .unique(subset=['id','value'], keep='first', maintain_order=True)
    .filter(pl.col("value").is_not_null())
    .filter(pl.col('id').is_in(ot_associations_diseases))
)
disease_names_df.head(2)

id,variable,value
str,str,str
"""DOID_0050890""","""name""","""synucleinopathy"""
"""DOID_10113""","""name""","""trypanosomiasis"""


In [10]:
disease_names_df.shape

(90643, 3)

In [11]:
disease_names_df['value'].is_null().any()

False

In [12]:
def load_model(model_name):
    with open("../models/"+model_name+".pkl","rb") as input:
        return pickle.load(input)

In [13]:
with open("../models/model_names.pkl","rb") as input:
    model_names = pickle.load(input)

print(model_names)

{'BioClinicalBERT': 'emilyalsentzer/Bio_ClinicalBERT', 'ClinicalBERT': 'medicalai/ClinicalBERT', 'BioBERT': 'dmis-lab/biobert-v1.1', 'BioClinical-ModernBERT': 'thomas-sounack/BioClinical-ModernBERT-base', 'SapBERT-from-PubMedBERT-fulltext': 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'}


In [14]:
embeddings_label = {}

#for model_name in ['']:

model_name = 'SapBERT-from-PubMedBERT-fulltext'
loaded_model = load_model(model_name)
model = loaded_model['model']
tokenizer = loaded_model['tokenizer']
input_texts = disease_names_df['value'].to_list()
embeddings_label[model_name] = eval_embeddings(
    model=model,
    tokenizer=tokenizer,
    input_texts=input_texts,
    pooling='mean',
    batch_size=32
)

cuda
Processing batches...
Using CLS representation...


  0%|          | 0/2833 [00:00<?, ?it/s]

In [15]:
disease_names_df = disease_names_df.with_columns(
    pl.DataFrame({"embeddings": list(embeddings_label[model_name])})
)

In [16]:
disease_names_df.write_parquet("../output/id_value_embeddings.parquet", compression="zstd")